# NEXTBUY: Feature Engineering

### Part 1: Import cleaned merged dataset

In [ ]:
import pandas as pd
from pathlib import Path

PROCESSED_PATH = Path("processed/")
parquet_file = PROCESSED_PATH / "full_data.parquet"
fallback_file = Path("full_data.parquet")

if parquet_file.exists():
    print("Loading from processed folder...")
    full_data = pd.read_parquet(parquet_file)
elif fallback_file.exists():
    print("Loading from current directory (fallback)...")
    full_data = pd.read_parquet(fallback_file)
else:
    raise FileNotFoundError(
        f"Processed data not found. "
        "Please run the data preprocessing notebook first."
    )

print(f"Loaded: {full_data.shape}")

In [ ]:
full_data.head()

### Part 2: User features

In [ ]:
# Order frequency
full_data["order_frequency"] = full_data.groupby("user_id")["days_since_prior_order"].transform("mean")


# Avg basket size
userStats = full_data.groupby("user_id")["order_id"].agg(["count", "nunique"])
avgBasketSize = userStats["count"] / userStats["nunique"]
full_data["avg_basket_size"] = full_data["user_id"].map(avgBasketSize)


# Preferred times
def getMode(series):
    mode = series.mode()
    if not mode.empty:
        return mode[0]
    else:
        return None

full_data["preferred_time_day-hour"] = full_data.groupby("user_id")["order_dow"].agg(getMode).astype(str) + "-" + full_data.groupby("user_id")["order_hour_of_day"].agg(getMode).astype(str)

### Part 3: Product features

In [ ]:
# Popularity
productModes = full_data.groupby("product_id")["product_id"].count().reset_index(name="Count").sort_values(["Count"], ascending=False)
full_data["product_buys_count"] = full_data["product_id"].map(productModes.set_index("product_id")["Count"])


# Reorder rate
reorderRates = full_data.groupby("product_id")["reordered"].mean().reset_index(name="Reorder Rate").sort_values(["Reorder Rate"], ascending=False)
full_data["product_reorder_rate"] = full_data["product_id"].map(reorderRates.set_index("product_id")["Reorder Rate"])

### Part 4: Data export

In [ ]:
full_data.head(50)

In [ ]:
import os

# Try processed folder first, fallback to current directory
processed_path = Path("processed")
output_file = processed_path / "full_data_engineered.parquet"

try:
    processed_path.mkdir(exist_ok=True)
    full_data.to_parquet(output_file, index=False)
    print(f"Saved to: {output_file}")
except Exception:
    # Fallback: save to current directory
    output_file = Path("full_data_engineered.parquet")
    full_data.to_parquet(output_file, index=False)
    print(f"Saved to: {output_file} (fallback - permission issue with processed folder)")

print(f"File size: {os.path.getsize(output_file) / 1e9:.2f} GB")